## Setup

`.env` must contain the following values:
```
SERVER_IP=your_server_ip
FTP_PORT=21
FTP_USER=your_username
FTP_PASSWORD=your_password
```

In [ ]:
from ftplib import FTP
from pathlib import Path
from dotenv import load_dotenv
import os

# --- LOAD ENV ---
BASE_DIR = Path().resolve().parents[2] / "Internomat"
env_path = BASE_DIR / ".env"

load_dotenv(env_path)

# --- ENV VARS ---
FTP_HOST = os.getenv("SERVER_IP")
FTP_PORT = int(os.getenv("FTP_PORT", 21))
FTP_USER = os.getenv("FTP_USER")
FTP_PASSWORD = os.getenv("FTP_PASSWORD")

print("\n[ENV DEBUG]")
print("FTP_HOST =", FTP_HOST)
print("FTP_PORT =", FTP_PORT)
print("FTP_USER =", FTP_USER)
print("FTP_PASSWORD =", "SET" if FTP_PASSWORD else "MISSING")

# --- HARD VALIDATION ---
assert FTP_HOST, "FTP_HOST missing (check .env)"
assert FTP_USER, "FTP_USER missing (check .env)"
assert FTP_PASSWORD, "FTP_PASSWORD missing (check .env)"

# --- PATHS ---
REMOTE_DIR = "/cs2/game/csgo/MatchZy"
LOCAL_DIR = BASE_DIR / "test_demos"  

LOCAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n[FTP] Connecting to {FTP_HOST}:{FTP_PORT}...")

# --- CONNECT ---
ftp = FTP()

try:
    ftp.connect(FTP_HOST, FTP_PORT, timeout=10)
    ftp.login(FTP_USER, FTP_PASSWORD)
    ftp.set_pasv(True) 

    print("[FTP] Connected")

    # --- NAVIGATE ---
    ftp.cwd(REMOTE_DIR)
    print(f"[FTP] Changed dir → {REMOTE_DIR}")

    files = ftp.nlst()

    downloaded = 0
    skipped = 0

    print(f"[FTP] Found {len(files)} files")

    for file in files:
        if not file.endswith(".dem"):
            continue

        local_file = LOCAL_DIR / file

        if local_file.exists():
            print(f"[SKIP] {file}")
            skipped += 1
            continue

        print(f"[DOWNLOAD] {file}")

        try:
            with open(local_file, "wb") as f:
                ftp.retrbinary(f"RETR {file}", f.write)
            downloaded += 1

        except Exception as e:
            print(f"[ERROR] Failed to download {file}: {e}")

    print("\n[FTP] Done")
    print(f"Downloaded: {downloaded}")
    print(f"Skipped: {skipped}")

except Exception as e:
    print(f"\n FTP ERROR: {e}")

finally:
    try:
        ftp.quit()
    except:
        pass

In [ ]:
from pathlib import Path
from awpy import Demo

DEMO_PATH = LOCAL_DIR  / "2026-03-20_21-57-48_3_de_mills_team_Sindilia_vs_team_Dottersack.dem"

print("Resolved path:", DEMO_PATH)
print("Exists:", DEMO_PATH.exists())


def load_demo(path: Path):
    path = path.resolve()

    if not path.exists():
        raise FileNotFoundError(f"Demo not found: {path}")

    print(f"[Demo] Loading: {path}")

    demo = Demo(path=str(path))
    demo.parse()

    print("[Demo] Parsing complete.")
    return demo


demo = load_demo(DEMO_PATH)

In [ ]:
def inspect_demo_attributes(demo):
    print("\n=== AVAILABLE DEMO ATTRIBUTES ===")

    attrs = dir(demo)

    # filter useful ones
    public_attrs = [a for a in attrs if not a.startswith("_")]

    for attr in sorted(public_attrs):
        try:
            value = getattr(demo, attr)

            if isinstance(value, list):
                print(f"{attr:20} -> list ({len(value)} items)")
            elif isinstance(value, dict):
                print(f"{attr:20} -> dict ({len(value)} keys)")
            else:
                print(f"{attr:20} -> {type(value).__name__}")

        except Exception as e:
            print(f"{attr:20} -> ERROR: {e}")


inspect_demo_attributes(demo)

In [ ]:
import pandas as pd

def parse_demo_full(demo):
    print("\n[Demo] Building structured tables (explicit)...")

    data = {}

    # --- HEADER ---
    data["header"] = demo.header

    # --- KNOWN TABLES (from your inspection) ---
    table_map = {
        "rounds": "rounds",
        "kills": "kills",
        "damages": "damages",
        "grenades": "grenades",
        "shots": "shots",
        "footsteps": "footsteps",
        "smokes": "smokes",
        "infernos": "infernos",
        "bomb": "bomb",
        "ticks": "ticks",
        "player_round_totals": "player_round_totals",
        "server_cvars": "server_cvars",
    }

    for key, attr in table_map.items():
        try:
            value = getattr(demo, attr, None)

            if value is None:
                print(f"[NULL] {key}")
                data[key] = None
                continue

            # force into DataFrame if possible
            if isinstance(value, pd.DataFrame):
                df = value
            elif isinstance(value, list):
                df = pd.DataFrame(value)
            else:
                df = value  # keep as-is (e.g. ticks may already be structured)

            data[key] = df

            if isinstance(df, pd.DataFrame):
                print(f"[OK]   {key:22} -> {len(df)} rows")
            else:
                print(f"[OK]   {key:22} -> {type(df).__name__}")

        except Exception as e:
            print(f"[FAIL] {key}: {e}")
            data[key] = None

    print("\n[Demo] Total collected:", len(data))

    return data


data = parse_demo_full(demo)
# print frame 
# print (data)

In [ ]:
def build_player_index(data):
    df = data["player_round_totals"]

    players = (
        df
        .filter(df["side"] == "all")
        .unique(subset=["steamid"])
    )

    print("[Players]", players.height)
    display(players)

    return players


players = build_player_index(data)

In [ ]:
import polars as pl

def build_player_stats(data):
    print("[Stats] Building...")

    kills = data["kills"]
    damages = data["damages"]
    shots = data["shots"]
    rounds = data["rounds"]

    # --- NORMALIZE INTO SAME SCHEMA ---
    kills_norm = (
        kills
        .select([
            pl.col("attacker_steamid").alias("steamid"),
            pl.lit(1).alias("kills"),
            pl.lit(0).alias("deaths"),
            pl.lit(0).alias("damage"),
            pl.lit(0).alias("shots"),
        ])
    )

    deaths_norm = (
        kills
        .select([
            pl.col("victim_steamid").alias("steamid"),
            pl.lit(0).alias("kills"),
            pl.lit(1).alias("deaths"),
            pl.lit(0).alias("damage"),
            pl.lit(0).alias("shots"),
        ])
    )

    damage_norm = (
        damages
        .select([
            pl.col("attacker_steamid").alias("steamid"),
            pl.lit(0).alias("kills"),
            pl.lit(0).alias("deaths"),
            pl.col("dmg_health_real").alias("damage"),
            pl.lit(0).alias("shots"),
        ])
    )

    shots_norm = (
        shots
        .select([
            pl.col("player_steamid").alias("steamid"),
            pl.lit(0).alias("kills"),
            pl.lit(0).alias("deaths"),
            pl.lit(0).alias("damage"),
            pl.lit(1).alias("shots"),
        ])
    )

    # --- CONCAT ALL EVENTS ---
    all_events = pl.concat([
        kills_norm,
        deaths_norm,
        damage_norm,
        shots_norm
    ])

    # --- AGGREGATE ---
    stats = (
        all_events
        .group_by("steamid")
        .agg([
            pl.sum("kills"),
            pl.sum("deaths"),
            pl.sum("damage"),
            pl.sum("shots"),
        ])
    )

    # --- DERIVED STATS ---
    n_rounds = rounds.height

    stats = stats.with_columns([
        (pl.col("kills") / pl.when(pl.col("deaths") == 0).then(1).otherwise(pl.col("deaths"))).alias("kd"),
        (pl.col("damage") / n_rounds).alias("adr"),
        (pl.col("kills") / pl.when(pl.col("shots") == 0).then(1).otherwise(pl.col("shots"))).alias("accuracy")
    ])

    stats = stats.sort("kills", descending=True)

    print("[Stats] Done")
    display(stats)

    return stats


player_stats = build_player_stats(data)